In [ ]:
# ============================================================
# CELL 1 — Check GPU (optional but convenient)
# ============================================================
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cpu


In [ ]:
# ============================================================
# CELL 2 — Load ESConv data (both files)
# ============================================================
import pandas as pd
import numpy as np

conv_df  = pd.read_csv("ESConv_Clean_Conversations.csv")
behav_df = pd.read_csv("behavioral_features_final.csv")   # from earlier notebook

print("Conversations shape:", conv_df.shape)
print("Behavioral features shape:", behav_df.shape)
print("\nConversations columns:", conv_df.columns.tolist())

Conversations shape: (908, 11)
Behavioral features shape: (910, 21)

Conversations columns: ['conversation_id', 'experience_type', 'emotion_type', 'problem_type', 'situation', 'initial_emotion_intensity', 'final_emotion_intensity', 'seeker_empathy_score', 'seeker_relevance_score', 'supporter_relevance_score', 'number_of_turns']


In [ ]:
# ============================================================
# CELL 3 — Check intensity distribution BEFORE deciding cutoff
# ============================================================
print("initial_emotion_intensity distribution:")
print(conv_df['initial_emotion_intensity'].value_counts().sort_index())
print("\nPercentage breakdown:")
print(conv_df['initial_emotion_intensity'].value_counts(normalize=True).sort_index() * 100)

initial_emotion_intensity distribution:
initial_emotion_intensity
1      2
2     31
3    173
4    400
5    302
Name: count, dtype: int64

Percentage breakdown:
initial_emotion_intensity
1     0.220264
2     3.414097
3    19.052863
4    44.052863
5    33.259912
Name: proportion, dtype: float64


In [ ]:
# ============================================================
# CELL 4 REVISED — Create binary label + check imbalance
# ============================================================
conv_df['distress_label'] = (
    conv_df['initial_emotion_intensity'] >= 4
).astype(int)

print("Distress label distribution:")
print(conv_df['distress_label'].value_counts())
print("\nProportion:")
print(conv_df['distress_label'].value_counts(normalize=True) * 100)

# Calculate class weights to handle imbalance
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels_array = conv_df['distress_label'].values
class_weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1]),
    y=labels_array
)
print("\nClass weights (for imbalance handling):", class_weights)
# Will show something like [2.21, 0.65] — LOW class gets weighted higher

Distress label distribution:
distress_label
1    702
0    206
Name: count, dtype: int64

Proportion:
distress_label
1    77.312775
0    22.687225
Name: proportion, dtype: float64

Class weights (for imbalance handling): [2.2038835  0.64672365]


In [ ]:
# ============================================================
# CELL 5 — Merge behavioral features with new label + situation text
# ============================================================
final_df = behav_df.merge(
    conv_df[['conversation_id', 'situation', 'distress_label',
             'emotion_type', 'problem_type']],
    on='conversation_id',
    how='left'
)

final_df = final_df.dropna(subset=['situation', 'distress_label'])

print("Final merged shape:", final_df.shape)
print("\nDistress label distribution in final data:")
print(final_df['distress_label'].value_counts())

Final merged shape: (908, 25)

Distress label distribution in final data:
distress_label
1.0    702
0.0    206
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL — Login using Colab's stored secret (run this FIRST)
# ============================================================
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('Hugging_face')   # use EXACT name shown in your secrets panel
login(token=hf_token)

print("Logged in successfully")

Logged in successfully


In [ ]:
# ============================================================
# CELL — Verify authentication
# ============================================================
from huggingface_hub import whoami

try:
    user_info = whoami()
    print("Authenticated as:", user_info['name'])
except Exception as e:
    print("Not authenticated:", e)

Authenticated as: pratik-2412


In [ ]:
# ============================================================
# CELL 6 — Extract text embeddings using MentalBERT (frozen encoder)
# ============================================================
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer_emb = AutoTokenizer.from_pretrained(
    "mental/mental-bert-base-uncased", token=True
)
model_emb = AutoModel.from_pretrained(
    "mental/mental-bert-base-uncased", token=True
)
model_emb.to(device)
model_emb.eval()   # freeze — no training, just extracting features

def get_cls_embedding(text):
    inputs = tokenizer_emb(
        str(text),
        return_tensors  = 'pt',
        truncation      = True,
        max_length      = 128,
        padding         = 'max_length'
    ).to(device)
    with torch.no_grad():
        output = model_emb(**inputs)
    return output.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

print("Extracting embeddings for", len(final_df), "conversations...")
embeddings = np.array([
    get_cls_embedding(text)
    for text in final_df['situation']
])
print("Embeddings shape:", embeddings.shape)   # (N, 768)

np.save("esconv_text_embeddings.npy", embeddings)
print("Saved esconv_text_embeddings.npy")

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] BertModel LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Extracting embeddings for 908 conversations...
Embeddings shape: (908, 768)
Saved esconv_text_embeddings.npy


In [ ]:
# Run this right after Cell 6 and Cell 8 complete successfully
np.save("esconv_text_embeddings.npy", embeddings)
np.save("X_behav.npy", X_behav)
np.save("y_labels.npy", y)
final_df.to_csv("final_df_with_labels.csv", index=False)
print("Checkpoint saved.")

Checkpoint saved.


In [ ]:
# ============================================================
# CELL 7 — Prepare behavioral features (same 8 features as before)
# ============================================================
from sklearn.preprocessing import StandardScaler

behavioral_cols = [
    'total_turn_count', 'engagement_ratio', 'avg_user_msg_length',
    'msg_length_slope', 'dropout_flag', 'dropout_gap',
    'user_msg_length_std', 'ended_by_user'
]

X_behav = final_df[behavioral_cols].fillna(0).values
scaler  = StandardScaler()
X_behav = scaler.fit_transform(X_behav)

print("Behavioral features shape:", X_behav.shape)

import joblib
joblib.dump(scaler, "behav_scaler_esconv.pkl")

Behavioral features shape: (908, 8)


['behav_scaler_esconv.pkl']

In [ ]:
# If NaNs exist, drop those rows before proceeding
final_df = final_df.dropna(subset=['distress_label']).reset_index(drop=True)

# Also re-slice embeddings and X_behav to match, since rows were dropped
# (only needed if NaNs were found)
print("Final_df shape after dropping NaN:", final_df.shape)

Final_df shape after dropping NaN: (908, 25)


In [ ]:
# ============================================================
# CELL 8 FIXED — Ensure labels are integers before bincount
# ============================================================
from sklearn.model_selection import train_test_split

# Force y to be integer type
y = final_df['distress_label'].astype(int).values

print("y dtype:", y.dtype)
print("Any NaN in y:", np.isnan(final_df['distress_label']).sum())

indices = np.arange(len(y))

idx_train, idx_temp = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.5, random_state=42, stratify=y[idx_temp]
)

print(f"Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")
print(f"\nTrain label distribution: {np.bincount(y[idx_train])}")
print(f"Val label distribution:   {np.bincount(y[idx_val])}")
print(f"Test label distribution:  {np.bincount(y[idx_test])}")

y dtype: int64
Any NaN in y: 0
Train: 726 | Val: 91 | Test: 91

Train label distribution: [165 561]
Val label distribution:   [20 71]
Test label distribution:  [21 70]


In [ ]:
# Run this right after Cell 6 and Cell 8 complete successfully
np.save("esconv_text_embeddings.npy", embeddings)
np.save("X_behav.npy", X_behav)
np.save("y_labels.npy", y)
final_df.to_csv("final_df_with_labels.csv", index=False)
print("Checkpoint saved.")

Checkpoint saved.


In [ ]:
# ============================================================
# CELL 9 — MODEL A: Text-only classifier
# ============================================================
import torch.nn as nn

class TextOnlyModel(nn.Module):
    def __init__(self, text_dim=768, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(text_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, text_emb):
        return self.net(text_emb)

model_A = TextOnlyModel().to(device)
print("Model A ready:", sum(p.numel() for p in model_A.parameters()), "params")

Model A ready: 98690 params


In [ ]:
# ============================================================
# CELL 10 REVISED — Train function with class weights applied
# ============================================================
from torch.optim import Adam

# Class weights from your output: [2.2038835, 0.64672365]
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print("Using class weights:", class_weights_tensor)

def train_simple_model(model, X_train, y_train, X_val, y_val,
                        epochs=30, lr=1e-3):
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)   # CHANGED
    optimizer = Adam(model.parameters(), lr=lr)

    X_train_t = torch.FloatTensor(X_train).to(device)
    y_train_t = torch.LongTensor(y_train).to(device)
    X_val_t   = torch.FloatTensor(X_val).to(device)
    y_val_t   = torch.LongTensor(y_val).to(device)

    best_val_acc = 0
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss    = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_t)
            val_preds   = val_outputs.argmax(dim=1)
            val_acc     = (val_preds == y_val_t).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}")

    print(f"\nBest Val Acc: {best_val_acc:.4f}")
    return model

print("Training Model A (Text Only)...")
model_A = train_simple_model(
    model_A,
    embeddings[idx_train], y[idx_train],
    embeddings[idx_val],   y[idx_val]
)

Using class weights: tensor([2.2039, 0.6467])
Training Model A (Text Only)...
Epoch 5/30 | Loss: 0.6790 | Val Acc: 0.3407
Epoch 10/30 | Loss: 0.6434 | Val Acc: 0.4945
Epoch 15/30 | Loss: 0.6260 | Val Acc: 0.5275
Epoch 20/30 | Loss: 0.6069 | Val Acc: 0.5055
Epoch 25/30 | Loss: 0.5785 | Val Acc: 0.5934
Epoch 30/30 | Loss: 0.5567 | Val Acc: 0.6374

Best Val Acc: 0.7802


In [ ]:
# ============================================================
# CELL 11 — MODEL B: Behavioral-only classifier
# ============================================================
class BehavioralOnlyModel(nn.Module):
    def __init__(self, behav_dim=8, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(behav_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, behav_feat):
        return self.net(behav_feat)

model_B = BehavioralOnlyModel().to(device)
print("Model B ready:", sum(p.numel() for p in model_B.parameters()), "params")

print("\nTraining Model B (Behavioral Only)...")
model_B = train_simple_model(
    model_B,
    X_behav[idx_train], y[idx_train],
    X_behav[idx_val],   y[idx_val]
)

Model B ready: 850 params

Training Model B (Behavioral Only)...
Epoch 5/30 | Loss: 0.6968 | Val Acc: 0.7033
Epoch 10/30 | Loss: 0.6939 | Val Acc: 0.6813
Epoch 15/30 | Loss: 0.6914 | Val Acc: 0.6813
Epoch 20/30 | Loss: 0.6892 | Val Acc: 0.6703
Epoch 25/30 | Loss: 0.6871 | Val Acc: 0.6264
Epoch 30/30 | Loss: 0.6850 | Val Acc: 0.5495

Best Val Acc: 0.7033


In [ ]:
# ============================================================
# CELL 12 — MODEL C: Fusion classifier
# ============================================================
class FusionModel(nn.Module):
    def __init__(self, text_dim=768, behav_dim=8, num_classes=2):
        super().__init__()
        self.text_branch = nn.Sequential(
            nn.Linear(text_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.behav_branch = nn.Sequential(
            nn.Linear(behav_dim, 32),
            nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, text_emb, behav_feat):
        t = self.text_branch(text_emb)
        b = self.behav_branch(behav_feat)
        combined = torch.cat([t, b], dim=1)
        return self.classifier(combined)

model_C = FusionModel().to(device)
print("Model C ready:", sum(p.numel() for p in model_C.parameters()), "params")

Model C ready: 109154 params


In [ ]:
import copy, random
import torch.nn as nn
from torch.optim import Adam

class FusionModelWeighted(nn.Module):
    def __init__(self, text_dim=768, behav_dim=8, num_classes=2):
        super().__init__()
        self.text_branch = nn.Sequential(
            nn.Linear(text_dim, 32), nn.ReLU(), nn.Dropout(0.4)
        )
        self.behav_branch = nn.Sequential(
            nn.Linear(behav_dim, 32), nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(16, num_classes)
        )

    def forward(self, text_emb, behav_feat):
        t = self.text_branch(text_emb)
        b = self.behav_branch(behav_feat)
        return self.classifier(torch.cat([t, b], dim=1))

# Reuse your existing TextOnlyModel and BehavioralOnlyModel definitions
print("Fusion (weighted) params:",
      sum(p.numel() for p in FusionModelWeighted().parameters()))

Fusion (weighted) params: 25970


In [ ]:
from sklearn.metrics import (balanced_accuracy_score, accuracy_score,
                             f1_score, precision_score, recall_score,
                             roc_auc_score)

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def train_model(model, inp_train, y_train, inp_val, y_val,
                epochs=60, lr=5e-4, weight_decay=1e-4):
    """inp_* are tuples of arrays: (text,), (behav,), or (text, behav)"""
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    Xtr = [torch.FloatTensor(x).to(device) for x in inp_train]
    Xva = [torch.FloatTensor(x).to(device) for x in inp_val]
    ytr = torch.LongTensor(y_train).to(device)

    best_score, best_state = -1, None
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(*Xtr), ytr)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            vpred = model(*Xva).argmax(dim=1).cpu().numpy()
        score = balanced_accuracy_score(y_val, vpred)
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)   # restore BEST epoch for every model
    return model

def evaluate(model, inp_test, y_test):
    model.eval()
    Xte = [torch.FloatTensor(x).to(device) for x in inp_test]
    with torch.no_grad():
        out = model(*Xte)
        prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        pred = out.argmax(dim=1).cpu().numpy()
    return {
        'acc':      accuracy_score(y_test, pred),
        'bal_acc':  balanced_accuracy_score(y_test, pred),
        'f1':       f1_score(y_test, pred, zero_division=0),
        'precision':precision_score(y_test, pred, zero_division=0),
        'recall':   recall_score(y_test, pred, zero_division=0),
        'auroc':    roc_auc_score(y_test, prob),
    }

builders = {
    'A - Text':       (lambda: TextOnlyModel(),        lambda i: (embeddings[i],)),
    'B - Behavioral': (lambda: BehavioralOnlyModel(),  lambda i: (X_behav[i],)),
    'C - Fusion':     (lambda: FusionModelWeighted(),  lambda i: (embeddings[i], X_behav[i])),
}

In [ ]:
import pandas as pd

seeds = [0, 1, 2, 3, 4]
rows = []
for seed in seeds:
    for name, (build, get_inp) in builders.items():
        set_seed(seed)
        m = build().to(device)
        m = train_model(m, get_inp(idx_train), y[idx_train],
                           get_inp(idx_val),   y[idx_val])
        r = evaluate(m, get_inp(idx_test), y[idx_test])
        r['model'] = name; r['seed'] = seed
        rows.append(r)

res = pd.DataFrame(rows)
summary = res.groupby('model')[['acc','bal_acc','f1','precision','recall','auroc']] \
             .agg(['mean','std']).round(3)
print(summary)

                  acc        bal_acc            f1        precision         \
                 mean    std    mean    std   mean    std      mean    std   
model                                                                        
A - Text        0.554  0.117   0.527  0.033  0.649  0.155     0.781  0.020   
B - Behavioral  0.448  0.030   0.385  0.018  0.583  0.035     0.695  0.016   
C - Fusion      0.576  0.035   0.514  0.035  0.694  0.034     0.777  0.020   

               recall         auroc         
                 mean    std   mean    std  
model                                       
A - Text        0.577  0.191  0.496  0.023  
B - Behavioral  0.503  0.043  0.370  0.022  
C - Fusion      0.629  0.049  0.477  0.025  


In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy.stats import ttest_rel

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []

for fold, (tr, te) in enumerate(skf.split(embeddings, y)):
    tr_i, va_i = train_test_split(tr, test_size=0.1,
                                  random_state=42, stratify=y[tr])
    for name, (build, get_inp) in builders.items():
        set_seed(fold)
        m = build().to(device)
        m = train_model(m, get_inp(tr_i), y[tr_i], get_inp(va_i), y[va_i])
        r = evaluate(m, get_inp(te), y[te])
        r['model'] = name; r['fold'] = fold
        cv_rows.append(r)

cv = pd.DataFrame(cv_rows)
print(cv.groupby('model')[['bal_acc','f1','auroc']].agg(['mean','std']).round(3))

a = cv[cv.model == 'A - Text'].sort_values('fold')['auroc'].values
c = cv[cv.model == 'C - Fusion'].sort_values('fold')['auroc'].values
t, p = ttest_rel(c, a)
print(f"\nAUROC  C vs A: mean diff = {np.mean(c - a):+.4f}, p = {p:.4f}")

               bal_acc            f1         auroc       
                  mean    std   mean    std   mean    std
model                                                    
A - Text         0.556  0.039  0.655  0.153  0.573  0.046
B - Behavioral   0.525  0.009  0.527  0.280  0.506  0.073
C - Fusion       0.549  0.043  0.710  0.052  0.559  0.052

AUROC  C vs A: mean diff = -0.0138, p = 0.2957


In [ ]:
# CELL 16 — Baselines + positive control (scaler is inside the pipeline: no leakage)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

skf = StratifiedKFold(5, shuffle=True, random_state=42)

def cv_scores(X, y_, C=0.1):
    clf = make_pipeline(StandardScaler(),
          LogisticRegression(C=C, class_weight='balanced', max_iter=3000))
    prob = cross_val_predict(clf, X, y_, cv=skf, method='predict_proba')[:, 1]
    return round(roc_auc_score(y_, prob), 3), round(balanced_accuracy_score(y_, prob > 0.5), 3)

print("Majority-class accuracy:", round(max(y.mean(), 1 - y.mean()), 3))
print("Text (LogReg)       AUROC, BalAcc:", cv_scores(embeddings, y))
print("Behavioral (LogReg) AUROC, BalAcc:", cv_scores(X_behav, y, C=1.0))

# Positive control: can the same embeddings predict emotion_type?
ok = final_df['emotion_type'].map(final_df['emotion_type'].value_counts()) >= 20
clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000))
f1m = cross_val_score(clf, embeddings[ok.values], final_df.loc[ok, 'emotion_type'],
                      cv=5, scoring='f1_macro')
print("Emotion_type macro-F1 from text:", f1m.mean().round(3))

Majority-class accuracy: 0.773
Text (LogReg)       AUROC, BalAcc: (np.float64(0.594), np.float64(0.582))
Behavioral (LogReg) AUROC, BalAcc: (np.float64(0.502), np.float64(0.509))


KeyError: 'emotion_type'

In [ ]:
# CELL — Check exactly what exists right now
import sys

check_vars = ['y', 'embeddings', 'X_behav', 'final_df', 'idx_train',
              'idx_val', 'idx_test', 'conv_df', 'behav_df', 'device']

for var in check_vars:
    exists = var in globals()
    if exists:
        val = globals()[var]
        info = f"shape={val.shape}" if hasattr(val, 'shape') else f"type={type(val)}"
        print(f"{var:15s}: EXISTS  ({info})")
    else:
        print(f"{var:15s}: MISSING")

y              : EXISTS  (shape=(908,))
embeddings     : EXISTS  (shape=(908, 768))
X_behav        : EXISTS  (shape=(908, 8))
final_df       : EXISTS  (shape=(908, 25))
idx_train      : EXISTS  (shape=(726,))
idx_val        : EXISTS  (shape=(91,))
idx_test       : EXISTS  (shape=(91,))
conv_df        : EXISTS  (shape=(908, 12))
behav_df       : EXISTS  (shape=(910, 21))
device         : EXISTS  (type=<class 'torch.device'>)


In [ ]:
# CELL — Check if checkpoint files exist on disk
import os

files_to_check = [
    "esconv_text_embeddings.npy",
    "X_behav.npy",
    "y_labels.npy",
    "final_df_with_labels.csv",
    "text_model_final.pt"
]

for f in files_to_check:
    exists = os.path.exists(f)
    size = f"{os.path.getsize(f)/1e6:.1f} MB" if exists else ""
    print(f"{f:35s}: {'FOUND ' + size if exists else 'NOT FOUND'}")

esconv_text_embeddings.npy         : FOUND 2.8 MB
X_behav.npy                        : FOUND 0.1 MB
y_labels.npy                       : FOUND 0.0 MB
final_df_with_labels.csv           : FOUND 0.3 MB
text_model_final.pt                : FOUND 3.1 MB
